In [2]:
from pathlib import Path
import pypandoc
from datetime import datetime
import re


# Where the pyproject.toml file is located.
BASE_DIR = Path().cwd().parent


INPUT_DIR = BASE_DIR.joinpath("part_a", "data", "generated_questions")

OUTPUT_DIR = BASE_DIR.joinpath("part_a", "data")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

now = datetime.now()
timestamp = now.strftime("%Y%m%d_%H%M%S")
OUTPUT_FILE_PATH = OUTPUT_DIR.joinpath(f"{timestamp}_generated_questions.docx")

def roman_to_int(roman):
    roman_numerals = {
        'M': 1000, 'CM': 900, 'D': 500, 'CD': 400,
        'C': 100, 'XC': 90, 'L': 50, 'XL': 40,
        'X': 10, 'IX': 9, 'V': 5, 'IV': 4, 'I': 1
    }
    i = 0
    num = 0
    while i < len(roman):
        if i+1 < len(roman) and roman[i:i+2] in roman_numerals:
            num += roman_numerals[roman[i:i+2]]
            i += 2
        else:
            num += roman_numerals[roman[i]]
            i += 1
    return num

def extract_roman_from_filename(filename):
    # Looks for 'título_xxx' or 'titulo_xxx' where xxx is a roman numeral
    match = re.search(r't[ií]tulo[_\s]?([ivxlcdm]+)', filename, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    return None

def order_files_by_roman(files):
    def sort_key(file):
        roman = extract_roman_from_filename(str(file))
        return roman_to_int(roman) if roman else float('inf')
    return sorted(files, key=sort_key)

def concatenate_and_convert_to_docx(input_dir, output_file_path):
    ordered_files = order_files_by_roman(list(input_dir.glob("*.md")))
    combined_md = []
    for file in ordered_files:
        header = f"## {file.name}\n"
        with open(file, 'r', encoding='utf-8') as f:
            content = f.read()
        combined_md.append(header + content + '\n')
    full_markdown = '\n'.join(combined_md)
    # Convert to docx using pypandoc
    pypandoc.convert_text(full_markdown, 'docx', format='md', outputfile=str(output_file_path))
    print(f"DOCX file created at: {output_file_path}")


In [4]:
try:
    concatenate_and_convert_to_docx(INPUT_DIR, OUTPUT_FILE_PATH)
except OSError as e:
    pypandoc.download_pandoc()
    concatenate_and_convert_to_docx(INPUT_DIR, OUTPUT_FILE_PATH)

DOCX file created at: /mnt/c/Users/leo24/OneDrive/ex_dropbox/tesis/examples/notebooks/part_a/data/20250804_101435_generated_questions.docx
